# Using the public methods of `OZITele`

This notebook gives a practical overview of the `OZITele` class defined in `OZITelemetry.py`.

It focuses on the following methods:

- `__init__`
- `compute_projectors`
- `extract_Zimages`
- `reconstruct_phase`
- `reconstruct_all_phase`
- `project_OPDs`
- `PSD_IFs`
- `PSD_modal`
- `compute_all_PSD`
- `PSD_cmd_IFs`
- `PSD_cmd_modal`

## Important remarks

This class depends on a specific local environment and external files/modules such as:

- `OOPAO`
- `plot_func`
- `Intensity_selection`
- `dm_second_stage_misreg_dict.npy`
- `IF_dm2.npy`

So this notebook is primarily an **explicit usage guide**.  
Some cells may require your full OZIRIIS/OOPAO environment to run successfully.


## 1. Create an `OZITele` object

The constructor loads the telemetry file, initializes the internal optical models,
extracts the command cubes, prepares the ZWFS images, and computes projector matrices automatically.

### Main parameters

- `tele_path`: `None` by default. If `None` a file selection window open. Otherwise it correspond to the path to the telemetry `.npy` file
- `is_onsky`: set `True` for on-sky telemetry, `False` for internal/calibration-like telemetry
- `CNN`: fallback flag only used if the reconstruction method is not found in the file

### Typical usage


In [1]:
from OZITelemetry import OZITele
import numpy as np
import matplotlib.pyplot as plt




   ✸       *          °          *      *                                      
        °   ✸         ▄██▄   ▄██▄  ▄███▄   ▄██▄ * ▄██▄    >           ▄▄▄▄     
  ✸            °     ██* ██ ██  ██ ██  ██ ██  ██ ██  ██   ===>     ▄█▀▀  ▀▀█▄  
   *   °    ✸        ██  ██ ██° ██ ██  ██ ██* ██ ██  ██   =>      █▀ ▄█▀▀█▄ ▀█ 
✸    *           °   ██  ██ ██  ██ ████▀  ██▄▄██ ██  ██   ====>  █▀ █▀ ▄▄ ▀█ ▀█
           ✸   °     ██* ██ ██  ██ ██     ██▀▀██ ██  ██   =====> █▄ █▄ ▀▀ ▄█ ▄█
 *    ✸     °        ██  ██ ██  ██ ██ *   ██  ██ ██* ██   =>      █▄ ▀█▄▄█▀ ▄█ 
    °        *    ✸   ▀██▀   ▀██▀  ██   ° ██  ██  ▀██▀    ==       ▀█▄▄  ▄▄█▀  
         ✸       *        *         *                                 ▀▀▀▀     




In [ ]:
tele_path =  "path/to/your/telemetry_file.npy" # None won't work on a notebook

tele = OZITele(
    tele_path=tele_path,
    is_onsky=True,
    CNN=False,
)

print("Telemetry loaded from:", tele.tele_path)
print("Image cube shape:", tele.img.shape)
print("ZWFS1 cube shape:", tele.img_ZWFS1.shape)
print("ZWFS2 cube shape:", tele.img_ZWFS2.shape)
print("Command cube shape:", tele.rec_cmd.shape)
print("Modal command cube shape:", tele.rec_cmd_modal.shape)


## 2. `compute_projectors()`

This method computes two internal projection matrices:

- `tele.proj_M2C`: projection onto modal coefficients
- `tele.proj_IF`: projection onto influence-function coefficients

### Important
This method is already called inside `__init__`, so you usually do **not** need to call it again.
You would only call it manually if you changed something in the telescope, DM, or modal basis after initialization.


In [ ]:
# Usually unnecessary because __init__ already does it
tele.compute_projectors()

print("Modal projector shape:", tele.proj_M2C.shape)
print("IF projector shape:", tele.proj_IF.shape)


## 3. `extract_Zimages()`

This method extracts the two ZWFS image streams from the normalized image cube,
rescales them when needed, and pads them to square arrays.

### Important
This method is also already called inside `__init__`.
Call it again only if you modified `tele.img`, masks, or pupil geometry and want to rebuild the ZWFS cubes.


In [ ]:
# Usually unnecessary because __init__ already does it
tele.extract_Zimages()

print("Updated ZWFS1 cube shape:", tele.img_ZWFS1.shape)
print("Updated ZWFS2 cube shape:", tele.img_ZWFS2.shape)


## 4. `reconstruct_phase(im1, im2, method='atan', damping=0.5, iteration=10)`

This reconstructs **one single phase map** from a pair of ZWFS images.

### Inputs

- `im1`: one frame from `tele.img_ZWFS1`
- `im2`: one frame from `tele.img_ZWFS2`
- `method`: reconstruction method, for example `"atan"`
- `damping`: damping factor for the iterative solver
- `iteration`: number of iterations

### Typical usage


In [ ]:
frame_id = 0

phase_map = tele.reconstruct_phase(
    tele.img_ZWFS1[frame_id],
    tele.img_ZWFS2[frame_id],
    method="atan",
    damping=0.5,
    iteration=10,
)

print("Single reconstructed phase shape:", phase_map.shape)

plt.figure(figsize=(5, 5))
plt.imshow(phase_map)
plt.title("Single reconstructed phase")
plt.colorbar()
plt.show()


## 5. `reconstruct_all_phase(method='atan', iteration=10, damping=0.5)`

This reconstructs the phase for the **full telemetry cube**.

### After running it
The object gains:

- `tele.phase`: reconstructed phase maps
- `tele.OPDs`: reconstructed OPDs in nm
- `tele.has_recontructed_phase = True`

### Typical usage


In [ ]:
tele.reconstruct_all_phase(
    method="atan",
    iteration=10,
    damping=0.5,
)

print("Phase cube shape:", tele.phase.shape)
print("OPD cube shape:", tele.OPDs.shape)
print("Has reconstructed phase:", tele.has_recontructed_phase)


## 6. `project_OPDs()`

This projects the reconstructed OPD maps onto:

- the influence-function basis: `tele.OPDs_on_IFs`
- the modal basis: `tele.OPDs_on_modes`

### Requirement
You must run `reconstruct_all_phase()` first.

### Typical usage


In [ ]:
tele.project_OPDs()

print("Projected OPDs on IFs shape:", tele.OPDs_on_IFs.shape)
print("Projected OPDs on modes shape:", tele.OPDs_on_modes.shape)
print("Has projected phase:", tele.has_projected_phase)


## 7. `PSD_IFs(npsg=None)`

This computes the PSD of the reconstructed OPDs after projection onto influence functions.

### Requirement
You must run:

1. `reconstruct_all_phase()`
2. `project_OPDs()`

### Output
The method creates:


- `tele.psd_IFs`
  
`npsg` is passed as `nperseg` to Welch's method.

Note that `tele.psd_IFs` is a tuple containing the freq vector and the PSD



In [ ]:
tele.PSD_IFs(npsg=1024)

print("PSD IFs object type:", type(tele.PSD_IFs))
print("PSD IFs size 0:", tele.PSD_IFs[0].size)
print("PSD IFs object type:", tele.PSD_IFs[1].size)



## 8. `PSD_modal(npsg=None)`

This computes the PSD of the reconstructed OPDs after projection onto modal coefficients.

### Requirement
You must run:

1. `reconstruct_all_phase()`
2. `project_OPDs()`

### Output
The method creates:

- `tele.psd_modal`

It works the same as IFs psd 

In [ ]:
tele.PSD_modal(npsg=1024)

print("PSD modal object type:", type(tele.psd_modal))
print("PSD modal size 0:", tele.PSD_modal[0].size)
print("PSD modal object type:", tele.PSD_modal[1].size)


## 9. `PSD_cmd_IFs(npsg=None)`

This computes the PSD of the real-time reconstructed command vectors in actuator space (`tele.rec_cmd`).

### Output
The method creates:

- `tele.psd_cmd_IFs`

Again, `tele.psd_cmd_IFs` returns `(frequencies, psd)`

In [ ]:
tele.PSD_cmd_IFs(npsg=1024)

print("PSD cmd IFs object type:", type(tele.psd_cmd_IFs))


## 10. `PSD_cmd_modal(npsg=None)`

This computes the PSD of the reconstructed command vectors in modal space (`tele.rec_cmd_modal`).

### Output
The method creates:

- `tele.psd_cmd_modal`


In [ ]:
tele.PSD_cmd_modal(npsg=1024)

print("PSD cmd modal object type:", type(tele.psd_cmd_modal))


## 11. `compute_all_PSD(npsg=None)`

This is the convenience method that computes **all PSD products** in sequence:

1. `PSD_IFs`
2. `PSD_modal`
3. `PSD_cmd_IFs`
4. `PSD_cmd_modal`

### Requirement
Since it calls `PSD_IFs` and `PSD_modal`, you still need reconstructed and projected OPDs first.


In [ ]:
tele.compute_all_PSD(npsg=1024)

print("All PSD computations completed.")


## 12. Recommended full workflow

In practice, the public methods are used in this order:


In [ ]:
tele = OZITele(tele_path="path/to/file.npy", is_onsky=True, CNN=False)

# Optional manual refreshes if you changed internals:
# tele.compute_projectors()
# tele.extract_Zimages()

# Reconstruct the full phase cube
tele.reconstruct_all_phase(method="atan", iteration=10, damping=0.5)

# Project reconstructed OPDs
tele.project_OPDs()

# Compute spectral products
tele.PSD_IFs(npsg=1024)
tele.PSD_modal(npsg=1024)
tele.PSD_cmd_IFs(npsg=1024)
tele.PSD_cmd_modal(npsg=1024)

# Or equivalently:
# tele.compute_all_PSD(npsg=1024)
